# Why attention needs positional embeddings

`x = [1, 2, 3, 4, 5]` and `y = [3, 1, 4, 2, 5]` contain the same tokens in a different order. Here attention only sees token embeddings, with no positional embedding.

- `y`'s attention scores are exactly `x`'s scores with the rows and columns reordered. For example, `y` starts with token 3, so its first row is `x`'s third row.
- The last position sees all five tokens in both orders, and gets an identical output: compare the last rows of `wx @ vx` and `wy @ vy`.
- Earlier positions differ only because the causal mask gives each one a different subset of tokens. Within that subset, order still doesn't matter.

So without positional embeddings, self-attention treats its context as a bag of tokens. Adding `pos_emb` to the token embeddings is what lets it tell orders apart (see `embedding_add`).

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [47]:
x = torch.tensor([[1,2,3,4,5]])
y = torch.tensor([[3,1,4,2,5]])
n_embd = 8
head_size = 2
block_size = 5

In [27]:
x.shape

torch.Size([1, 5])

In [48]:
tet = nn.Embedding(6, n_embd)
key = nn.Linear(n_embd, head_size, bias=False)
query = nn.Linear(n_embd, head_size, bias=False)

In [49]:
kx = key(tet(x))

In [50]:
ky = key(tet(y))

In [51]:
qx = query(tet(x))

In [52]:
qy = query(tet(y))

In [55]:
tril = torch.tril(torch.ones(block_size, block_size))

In [53]:
print((qx @ kx.transpose(-2,-1)))
print((qx @ kx.transpose(-2,-1)).shape)

tensor([[[ 1.5083e-01, -6.0618e-02,  2.9019e-02,  2.2997e-01,  3.4520e-02],
         [ 1.8846e-01,  3.4018e-02,  2.9865e-01,  1.9413e-01,  1.3796e-01],
         [ 3.5864e-01,  1.4687e-01,  7.6470e-01,  2.9969e-01,  3.3351e-01],
         [-5.6033e-02, -7.7267e-02, -2.4933e-01, -6.9750e-04, -9.9038e-02],
         [ 5.9678e-02, -3.2410e-02, -8.6625e-03,  9.8143e-02,  6.3781e-03]]],
       grad_fn=<UnsafeViewBackward0>)
torch.Size([1, 5, 5])


In [54]:
print((qy @ ky.transpose(-2,-1)))
print((qy @ ky.transpose(-2,-1)).shape)

tensor([[[ 7.6470e-01,  3.5864e-01,  2.9969e-01,  1.4687e-01,  3.3351e-01],
         [ 2.9019e-02,  1.5083e-01,  2.2997e-01, -6.0618e-02,  3.4520e-02],
         [-2.4933e-01, -5.6033e-02, -6.9750e-04, -7.7267e-02, -9.9038e-02],
         [ 2.9865e-01,  1.8846e-01,  1.9413e-01,  3.4018e-02,  1.3796e-01],
         [-8.6625e-03,  5.9678e-02,  9.8143e-02, -3.2410e-02,  6.3781e-03]]],
       grad_fn=<UnsafeViewBackward0>)
torch.Size([1, 5, 5])


In [57]:
wx = (qx @ kx.transpose(-2,-1))
wx = wx.masked_fill(tril[:block_size, :block_size] == 0, float("-inf"))
wx = F.softmax(wx, dim=-1)

In [58]:
wy = (qy @ ky.transpose(-2,-1))
wy = wy.masked_fill(tril[:block_size, :block_size] == 0, float("-inf"))
wy = F.softmax(wy, dim=-1)

In [59]:
wx

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5385, 0.4615, 0.0000, 0.0000, 0.0000],
         [0.3021, 0.2445, 0.4534, 0.0000, 0.0000],
         [0.2591, 0.2536, 0.2135, 0.2738, 0.0000],
         [0.2069, 0.1887, 0.1932, 0.2150, 0.1962]]],
       grad_fn=<SoftmaxBackward0>)

In [60]:
wy

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4696, 0.5304, 0.0000, 0.0000, 0.0000],
         [0.2861, 0.3471, 0.3668, 0.0000, 0.0000],
         [0.2806, 0.2513, 0.2527, 0.2154, 0.0000],
         [0.1932, 0.2069, 0.2150, 0.1887, 0.1962]]],
       grad_fn=<SoftmaxBackward0>)

In [61]:
value = nn.Linear(n_embd, head_size, bias=False)

In [63]:
vx = value(tet(x))
vy = value(tet(y))

In [67]:
wx @ vx

tensor([[[ 0.0402, -0.2219],
         [-0.0656, -0.4280],
         [-0.0481, -0.4025],
         [-0.0722, -0.4343],
         [ 0.0392, -0.4418]]], grad_fn=<UnsafeViewBackward0>)

In [68]:
wy @ vy

tensor([[[-0.0309, -0.3795],
         [ 0.0068, -0.2959],
         [-0.0325, -0.3547],
         [-0.0652, -0.4228],
         [ 0.0392, -0.4418]]], grad_fn=<UnsafeViewBackward0>)